# Tokenization in Large Language Models (LLMs)

Tokenization is the foundational step in processing text for Large Language Models (LLMs). Before feeding text into neural architectures like Transformers or RNNs, continuous running text must be segmented into discrete units called **tokens**.

Early approaches used simple delimiters (spaces, tabs, punctuation) to split text into words, but modern LLMs require robust strategies to handle out-of-vocabulary (OOV) words, complex morphology, and memory efficiency.

## 1. The Tokenization Spectrum

There are three primary paradigms for tokenization, each with distinct trade-offs:

1. **Word-Level Tokenization:**
* **Concept:** Splitting text into whole words.
* **Problem:** Generates an exponentially large vocabulary. It fails completely on unknown words (e.g., if "catch" is in the training set but "catching" is not, the model cannot process "catching").


2. **Character-Level Tokenization:**
* **Concept:** Splitting text into individual characters (e.g., 26 English letters + special symbols).
* **Problem:** Extremely limited vocabulary, but sequence lengths become excessively long. Characters alone lack inherent semantic meaning compared to words.


3. **Subword-Level Tokenization (The Modern Standard):**
* **Concept:** A middle ground that breaks words into meaningful subwords (e.g., "catching" $\rightarrow$ "catch" + "ing", or "hugging" $\rightarrow$ "hug" + "ing").
* **Advantage:** If the model has learned the subwords "catch" and "ing" during training, it can successfully parse the unseen word "catching" during testing and infer semantic similarity to words like "hugging".



Modern subword tokenizers consist of two separate modules:

* **Token Learner (Training Time):** Processes the training corpus to identify and construct the vocabulary.
* **Token Segmenter (Test/Inference Time):** Uses the learned vocabulary (and sometimes learned rules) to segment new, unseen text.

---

## 2. Byte Pair Encoding (BPE)

**Used in:** GPT, GPT-2, RoBERTa, BART.
Originally developed as a text compression algorithm, BPE was adapted for Neural Machine Translation in 2016. It builds a vocabulary from the bottom up based on **frequency**.

### The BPE Algorithm (Token Learner)

1. **Pre-tokenization:** Clean the corpus (remove punctuation, normalize case) and split sentences into initial words using spaces/rules.
2. **Initialize Base Vocabulary:** Extract all unique characters present in the corpus. For example, the word "cat" adds `c`, `a`, and `t` to the base vocabulary.
3. **Count Frequencies:** Count the occurrences of every word in the training set.
4. **Merge by Frequency:**
* Calculate the frequency of all adjacent token pairs (bigrams).
* Identify the most frequent pair.
* Merge them into a new single token and add it to the vocabulary.
* Record this **merge rule** (e.g., `a` + `t` $\rightarrow$ `at`).


5. **Iterate:** Repeat the merging process until a predefined vocabulary size threshold (e.g., 10,000 tokens) is reached.

### Test Time (Token Segmenter)

During inference, when faced with an unknown word (e.g., "BAGS"):

1. Split the word into base characters: `B`, `A`, `G`, `S`.
2. Apply the saved merge rules in the **exact sequential order** they were learned during training.
3. If a character is completely missing from the learned base vocabulary (e.g., the letter `M` in "MAT"), replace only that specific unknown character with the special `<unk>` token. The final segmentation might look like `<unk>`, `AT`.

---

## 3. WordPiece Tokenization

**Used in:** BERT, DistilBERT, MobileBERT.
WordPiece operates similarly to BPE (bottom-up approach) but differs fundamentally in its **merging criteria**, **internal notation**, and **test-time segmentation**.

### The WordPiece Algorithm (Token Learner)

1. **Subword Notation (`##`):** WordPiece distinguishes between characters at the *start* of a word and characters in the *middle*.
* "HELLO" is initially split into: `H`, `##E`, `##L`, `##L`, `##O`.
* The `##` prefix indicates that the token is attached to the previous token.


2. **Merge by Likelihood (Not just frequency):** Instead of merging the most frequent pair, WordPiece merges the pair that maximizes the likelihood of the training data. It uses a Pointwise Mutual Information (PMI)-like score:
$$Score = \frac{\text{Frequency}(CD)}{\text{Frequency}(C) \times \text{Frequency}(D)}$$


* **Why this formula?** It penalizes highly frequent individual characters. A pair is given high priority if they appear together frequently *relative* to how often they appear individually.



### Test Time (Token Segmenter)

**Crucial Distinction:** WordPiece **does not store merge rules**. It only stores the final learned vocabulary.

1. Given a test word (e.g., "fused"), it uses a **Longest Subword Matching** strategy.
2. It scans from the beginning of the word and searches the vocabulary for the longest possible valid prefix (e.g., `f`, then checks `fu`, etc.).
3. Once the longest prefix is found, it segments it, and repeats the process for the remaining substring (which must now match tokens with the `##` prefix).
4. **OOV Handling:** If at any point a substring cannot be matched to *any* token in the vocabulary, the **entire word** is replaced by `<unk>`. (Contrast this with BPE, which only replaces the specific unknown character).

---

## 4. Unigram Language Model Tokenization (SentencePiece)

**Used in:** T5, XLNet, ALBERT (often via SentencePiece).
Unlike BPE and WordPiece which build the vocabulary from the bottom up, the Unigram method is a **top-down (shrinking)** approach.

### Concept & Mathematics

It relies on the Unigram Language Model assumption, where the probability of a sequence of tokens is the product of their independent probabilities:

$$P(w_1, w_2, \dots, w_n) = P(w_1) \times P(w_2) \dots \times P(w_n)$$

### The Unigram Algorithm (Token Learner)

1. **Initialize a Massive Vocabulary:** Start with a huge vocabulary containing all characters, common substrings, and words (sometimes initialized by running BPE exhaustively).
2. **Calculate Unigram Probabilities:** For every token in the vocabulary, calculate its probability: $P(\text{token}) = \frac{\text{freq(token)}}{\text{total tokens}}$.
3. **Find the Best Split (Viterbi):** A word like "run" could be split as `r, u, n` or `ru, n` or `r, un`. Using the **Viterbi algorithm** (for computational efficiency), find the split that yields the highest probability based on the current vocabulary.
4. **Compute Log-Likelihood Loss:** For the entire training corpus, calculate the total loss based on the best splits:
$$\text{Loss} = \sum_{\text{corpus}} \left( \text{Frequency of word} \times -\log(P(\text{best split})) \right)$$


5. **Pruning (Expectation-Maximization):**
* Temporarily remove tokens one by one and recalculate the total loss.
* Identify tokens whose removal causes the *least* increase in the total loss.
* Permanently drop the bottom 10% to 20% of these "useless" tokens.


6. **Iterate:** Repeat the probability calculation and pruning steps until the target vocabulary size is reached.

---

## Exam-Oriented Practice Questions

**Q1: During test time, an LLM encounters the word "Zebra". The character "Z" was never seen in the training corpus. How will BPE and WordPiece handle this respectively?**
A) Both will replace the entire word with `<unk>`.
B) BPE will output `<unk>, e, b, r, a`. WordPiece will output `<unk>`.
C) BPE will output `<unk>`. WordPiece will output `<unk>, ##e, ##b, ##r, ##a`.
D) Both will ignore the "Z" and output `e, b, r, a`.

* **Answer & Explanation:** **B**. BPE replaces only the specific unknown character with `<unk>` and continues segmenting the rest based on stored rules. WordPiece operates strictly on longest-match vocabulary lookup; if any part of the word fails to match, the *entire word* is discarded and replaced with `<unk>`.

**Q2: In WordPiece tokenization, if the frequency of token `A` is 10, token `B` is 5, and the frequency of the pair `AB` occurring together is 2, what is the merge score for `AB`?**
A) 0.04
B) 25
C) 0.4
D) 50

* **Answer & Explanation:** **A**. The WordPiece score is calculated as $\frac{\text{freq}(AB)}{\text{freq}(A) \times \text{freq}(B)}$. Here, $2 / (10 \times 5) = 2 / 50 = 0.04$.

**Q3: Which of the following tokenization strategies starts with a massive base vocabulary and iteratively shrinks it?**
A) Byte Pair Encoding (BPE)
B) WordPiece
C) FastText
D) Unigram Language Model Tokenization

* **Answer & Explanation:** **D**. BPE and WordPiece are bottom-up approaches (start with characters, merge to build up). Unigram is top-down; it starts with all possible subwords and prunes the ones that impact the log-loss the least.

**Q4: Why does Unigram tokenization use the Viterbi algorithm during training?**
A) To clean the corpus of punctuation.
B) To calculate the Pointwise Mutual Information of characters.
C) To efficiently find the highest probability segmentation among all possible combinations of subwords for a given word.
D) To replace the Expectation-Maximization (EM) step.

* **Answer & Explanation:** **C**. Because a single word can be split in many valid ways based on a large vocabulary, exhaustively checking every combination is too computationally expensive. Viterbi decoding efficiently finds the maximum probability path (split).

---

## Quick Revision Table

| Feature | Byte Pair Encoding (BPE) | WordPiece | Unigram (SentencePiece) |
| --- | --- | --- | --- |
| **Primary Models** | GPT, GPT-2, RoBERTa, BART | BERT, DistilBERT, MobileBERT | T5, XLNet, ALBERT |
| **Building Direction** | Bottom-Up (Grows vocabulary) | Bottom-Up (Grows vocabulary) | Top-Down (Shrinks vocabulary) |
| **Merge Criteria** | Highest Frequency | Highest Likelihood (Score) | Minimizes Log-Likelihood Loss |
| **Internal Prefixes** | None | `##` for internal characters | None (uses raw probabilities) |
| **Test-Time Strategy** | Applies saved merge rules in order | Longest subword matching (no rules stored) | Finds split with highest unigram probability (Viterbi) |
| **OOV Handling** | Replaces missing *characters* with `<unk>` | Replaces the *entire word* with `<unk>` | Assigns low probability / `<unk>` mapping |